In [1]:
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments
)

In [2]:
df = pd.read_json(
    r"C:\Users\EileneAnnaKuriakose\Desktop\databricks-dolly-15k.jsonl",
    lines=True
)

In [3]:
df.head()

,instruction,context,response,category
0,When did Virgin Australia start operating?,"Virgin Australia, the trading name of Virgin A...",Virgin Australia commenced services on 31 Augu...,closed_qa
1,Which is a species of fish? Tope or Rope,,Tope,classification
2,Why can camels survive for long without water?,,Camels use the fat in their humps to keep them...,open_qa
3,"Alice's parents have three daughters: Amy, Jes...",,The name of the third daughter is Alice,open_qa
4,When was Tomoaki Komorida born?,Komorida was born in Kumamoto Prefecture on Ju...,"Tomoaki Komorida was born on July 10,1981.",closed_qa


In [25]:
df["text"] = (
    "Instruction: " + df["instruction"] +
    "\nContext: " + df["context"].fillna("") +
    "\nResponse: " + df["response"]
)

In [26]:
df = df[["text"]]

In [27]:
dataset = Dataset.from_pandas(df)

In [28]:
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [29]:
train_dataset = train_dataset.select(range(100))
test_dataset = test_dataset.select(range(20))

In [30]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small")

In [31]:
tokenizer.pad_token = tokenizer.eos_token

In [32]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [33]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [34]:
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/DialoGPT-small"
)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

In [35]:
training_args = TrainingArguments(
    output_dir="./dialogpt_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

In [36]:
def add_labels(example):
    example["labels"] = example["input_ids"].copy()
    return example

train_dataset = train_dataset.map(add_labels)
test_dataset = test_dataset.map(add_labels)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [37]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
C:\Users\EileneAnnaKuriakose\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
